# Apache Iceberg com Apache Spark

Este notebook demonstra como usar Apache Iceberg com PySpark para criar uma tabela local.

- Criação de catálogo Iceberg local
- Criação de tabela Iceberg
- Inserção e leitura de dados
- Evolução de schema

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

warehouse = os.path.abspath("tmp/iceberg_warehouse")
os.makedirs(warehouse, exist_ok=True)

spark = (
    SparkSession.builder
        .appName("IcebergStudy")
        .master("local[*]")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.local.type", "hadoop")
        .config("spark.sql.catalog.local.warehouse", warehouse)
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0")
        .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.default")

AttributeError: 'Builder' object has no attribute 'sql'

In [2]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS local.default.iceberg_events (
  id INT,
  name STRING,
  event_date DATE
)
USING iceberg
"""

spark.sql(create_table_sql)

data = [
    (1, "Alice", "2024-04-01"),
    (2, "Bob", "2024-04-02"),
    (3, "Carol", "2024-04-03")
]
df = spark.createDataFrame(data, ["id", "name", "event_date"]) \
    .withColumn("event_date", F.to_date("event_date"))

df.write.format("iceberg").mode("append").save("local.default.iceberg_events")

spark.sql("SELECT * FROM local.default.iceberg_events").show()

NameError: name 'spark' is not defined

In [3]:
spark.sql("ALTER TABLE local.default.iceberg_events ADD COLUMN country STRING")

new_data = [
    (4, "Diego", "2024-04-04", "BR"),
]
df2 = spark.createDataFrame(new_data, ["id", "name", "event_date", "country"]) \
    .withColumn("event_date", F.to_date("event_date"))

df2.write.format("iceberg").mode("append").save("local.default.iceberg_events")

spark.sql("SELECT * FROM local.default.iceberg_events").show()

spark.sql("DESCRIBE local.default.iceberg_events").show(truncate=False)

NameError: name 'spark' is not defined